# Cosmos DB Connection Test
Run cells top to bottom. Each cell is isolated so you can see exactly where it fails.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]   # COLEPV1
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from colep_ai.core.config import settings

In [3]:
# Cell 1 — paste your credentials directly here to rule out .env issues
COSMOS_URL = settings.COSMOS_URL
COSMOS_KEY = settings.COSMOS_KEY.get_secret_value()
DB_NAME = "colep_ai"
CONTAINER_NAME = "chat_sessions"

# Sanity checks on the key before even hitting the network
print(f"URL: {COSMOS_URL[:6]}")
print(f"Key length: {len(COSMOS_KEY)} chars")   # should be 88 chars for a standard Cosmos key
print(f"Key starts with: {COSMOS_KEY[:6]}...")
print(f"Key ends with: ...{COSMOS_KEY[-6:]}")
print(f"Has whitespace/newline: {any(c in COSMOS_KEY for c in [' ', '\\n', '\\r', '\\t'])}")

URL: https:
Key length: 88 chars
Key starts with: mYaJeH...
Key ends with: ...rNjg==
Has whitespace/newline: False


In [4]:
# Cell 2 — test with sync client first (simpler, no event loop complexity)
from azure.cosmos import CosmosClient

client = CosmosClient(url=COSMOS_URL, credential=COSMOS_KEY)

# list_databases is the lightest possible authenticated call
dbs = list(client.list_databases())
print(f"Connected. Databases found: {[d['id'] for d in dbs]}")

Connected. Databases found: []


In [5]:
# Cell 3 — create DB and container (only runs if Cell 2 passed)
db = client.create_database_if_not_exists(id=DB_NAME)
print(f"Database ready: {db.id}")

from azure.cosmos import PartitionKey
container = db.create_container_if_not_exists(
    id=CONTAINER_NAME,
    partition_key=PartitionKey(path="/session_id"),
)
print(f"Container ready: {container.id}")

Database ready: colep_ai
Container ready: chat_sessions


In [6]:
# Cell 4 — write a test document
test_doc = {
    "id": "test_session_001",
    "session_id": "test_session_001",
    "doc_type": "session",
    "user_id": "test_user",
    "user_name": "Test User",
    "summary": "",
    "created_at": "2026-07-31T00:00:00+00:00",
    "updated_at": "2026-07-31T00:00:00+00:00",
}
container.upsert_item(body=test_doc)
print("Write OK")

Write OK


In [7]:
# Cell 5 — read it back
doc = container.read_item(item="test_session_001", partition_key="test_session_001")
print(f"Read OK: {doc}")

Read OK: {'id': 'test_session_001', 'session_id': 'test_session_001', 'doc_type': 'session', 'user_id': 'test_user', 'user_name': 'Test User', 'summary': '', 'created_at': '2026-07-31T00:00:00+00:00', 'updated_at': '2026-07-31T00:00:00+00:00', '_rid': 'z4BvAPZ7lK0BAAAAAAAAAA==', '_self': 'dbs/z4BvAA==/colls/z4BvAPZ7lK0=/docs/z4BvAPZ7lK0BAAAAAAAAAA==/', '_etag': '"16000237-0000-0800-0000-6a704cf10000"', '_attachments': 'attachments/', '_ts': 1785744625}


In [8]:
# Cell 6 — cleanup test document
container.delete_item(item="test_session_001", partition_key="test_session_001")
print("Cleanup OK — test document deleted")

Cleanup OK — test document deleted


In [9]:
# Cell 7 — async client test (only run after sync passes)
import asyncio
from azure.cosmos.aio import CosmosClient as AsyncCosmosClient

async def test_async():
    async with AsyncCosmosClient(url=COSMOS_URL, credential=COSMOS_KEY) as client:
        container = client.get_database_client(DB_NAME).get_container_client(CONTAINER_NAME)
        doc = {
            "id": "test_async_001",
            "session_id": "test_async_001",
            "doc_type": "session",
            "summary": "",
        }
        await container.upsert_item(body=doc)
        result = await container.read_item(item="test_async_001", partition_key="test_async_001")
        print(f"Async read OK: {result['id']}")
        await container.delete_item(item="test_async_001", partition_key="test_async_001")
        print("Async cleanup OK")

await test_async()

Async read OK: test_async_001
Async cleanup OK
